# OTAKU Upscale V1 🔥

Simple workflow: **Upload → Upscale → Preview → Download**

Powered by Real-ESRGAN.

In [ ]:
!nvidia-smi


In [ ]:
!apt-get -qq update
!apt-get -qq install python3.11 python3.11-venv
!python3.11 -m venv /content/venv311
!/content/venv311/bin/python -m ensurepip --upgrade
!/content/venv311/bin/python -m pip install --upgrade pip
!/content/venv311/bin/pip install -q basicsr facexlib gfpgan
%cd /content
!rm -rf /content/realesrgan
!git clone -q https://github.com/xinntao/Real-ESRGAN.git realesrgan
!sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /content/venv311/lib/python3.11/site-packages/basicsr/data/degradations.py
print('Environment ready ✅')


In [ ]:
%%writefile /content/realesrgan/realesrgan/version.py
__version__ = "0.3.0"


In [ ]:
!mkdir -p /content/realesrgan/weights
!rm -f /content/realesrgan/weights/RealESRGAN_x4plus.pth
!wget -q --show-progress https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -O /content/realesrgan/weights/RealESRGAN_x4plus.pth
!ls -lh /content/realesrgan/weights/RealESRGAN_x4plus.pth
print('Model ready ✅')


In [ ]:
from google.colab import files
from pathlib import Path

input_folder = Path('/content/input')
output_folder = Path('/content/output')
input_folder.mkdir(exist_ok=True)
output_folder.mkdir(exist_ok=True)

uploaded = files.upload()
for filename, data in uploaded.items():
    (input_folder / filename).write_bytes(data)

print('\nUploaded:')
for filename in uploaded:
    print('✓', filename)


In [ ]:
import subprocess
from pathlib import Path

image_extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
input_files = [p for p in input_folder.iterdir() if p.suffix.lower() in image_extensions]

if not input_files:
    raise Exception('No supported image found. Run the upload cell first.')

for p in output_folder.iterdir():
    if p.is_file():
        p.unlink()

input_file = input_files[0]

command = [
    '/content/venv311/bin/python',
    '/content/realesrgan/inference_realesrgan.py',
    '-n', 'RealESRGAN_x4plus',
    '-i', str(input_file),
    '-o', str(output_folder),
    '--outscale', '4',
    '--model_path', '/content/realesrgan/weights/RealESRGAN_x4plus.pth'
]

print(f'Upscaling: {input_file.name}')
result = subprocess.run(command, capture_output=True, text=True)
print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise Exception('Upscaling failed.')

print('🔥 UPSCALE COMPLETE')


In [ ]:
from IPython.display import display
from PIL import Image

output_files = [p for p in output_folder.iterdir() if p.is_file()]

if not output_files:
    raise Exception('No output found.')

for file in output_files:
    img = Image.open(file)
    print(f'Output: {file.name} | Size: {img.size} | Format: {img.format}')
    display(img)


In [ ]:
from google.colab import files

output_files = [p for p in output_folder.iterdir() if p.is_file()]

if output_files:
    files.download(str(output_files[0]))
else:
    print('No output file found.')


## V1 status

✅ Colab GPU  
✅ Python 3.11 compatibility  
✅ Real-ESRGAN x4plus  
✅ Image upload  
✅ 4× AI upscale  
✅ Preview  
✅ Manual download

### Planned later
- Video upscaling
- Anime-specific processing
- 60 FPS interpolation
- CC / effects
- Oni Studio UI